# A framework to simultaneously optimize the material and energy transition


__author__ = "Yilun Lin"
__copyright__ = "Copyright 2023, Multi-parametric Optimization & Control Lab"
__credits__ = ["Yilun Lin", "Rahul Kakodkar", "Efstratios N. Pistikopoulos"]
__license__ = "Open"
__version__ = "1.0.0"
__maintainer__ = "Rahul Kakodkar"
__email__ = "cacodcar@tamu.edu"
__status__ = "Production"


In the following case study, we compare different technology options for the energy transition.

Each technology option also includes the different materials that are required for their establishement, as also their associated global warming and toxicity potentials.

**bold**

*italics*

# large

Material options for different types of wind mills, batteries

$\textbf{Import modules}$

In [1]:
import pandas 
from numpy import poly1d, polyfit, arange
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result 
from src.energiapy.model.formulate import formulate, Constraints, Objective
from src.energiapy.utils.data_utils import get_data, make_henry_price_df
from src.energiapy.utils.nsrdb_utils import fetch_nsrdb_data
from src.energiapy.plot import plot
from src.energiapy.model.solve import solve
from src.energiapy.utils.cluster_utils import reduce_scenario, agg_hierarchial_elbow,\
    Clustermethod, dynamic_warping, dynamic_warping_matrix, dynamic_warping_path
from src.energiapy.utils.data_utils import load_results
import matplotlib.pyplot as plt
from matplotlib import rc
from itertools import product

**Import solar dni and wind speeds for Harris county**

In [2]:
weather20_df = pandas.read_csv('data/ho_solar20.csv', index_col=0)
weather20_df.index = [i.split('+')[0] for i in weather20_df.index]
weather = weather20_df[~weather20_df.index.str.contains('02-29')] #remove leap years
weather

,wind_speed,dni
2020-01-01 00:00:00,9.5,0.0
2020-01-01 01:00:00,7.5,0.0
2020-01-01 02:00:00,6.0,0.0
2020-01-01 03:00:00,6.0,0.0
2020-01-01 04:00:00,6.0,0.0
...,...,...
2020-12-31 19:00:00,54.5,0.5
2020-12-31 20:00:00,55.5,0.0
2020-12-31 21:00:00,50.0,0.0
2020-12-31 22:00:00,46.0,181.5


**Demand data from ERCOT**

In [3]:
ercot20 = pandas.read_excel('data/Native_Load_2020.xlsx')
ercot = ercot20[['COAST']]
ercot['index'] = weather20_df.index
ercot = ercot.set_index('index')
ercot = ercot[~ercot.index.str.contains('02-29')]
ercot

/var/folders/w4/67h4yjmj6dj3l3gp3sz0hbqr0000gn/T/ipykernel_42031/4195252575.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ercot['index'] = weather20_df.index


,COAST
index,
2020-01-01 00:00:00,9890.091522
2020-01-01 01:00:00,9751.586415
2020-01-01 02:00:00,9603.421087
2020-01-01 03:00:00,9501.585599
2020-01-01 04:00:00,9499.426925
...,...
2020-12-31 19:00:00,12068.071946
2020-12-31 20:00:00,11818.903690
2020-12-31 21:00:00,11560.408167


**Import natural gas prices from Henry Hub Price Index**  

In [4]:
#Actual temporal scale (daily)
ng_price20 = make_henry_price_df(
    file_name='data/Henry_Hub_Natural_Gas_Spot_Price_Daily.csv', year=2020, stretch=False)
ng_price_df = ng_price20
ng_price_df['index'] = [i for i in weather.index][::24]
ng_price_df = ng_price_df.drop(columns= 'scales')
ng_price = ng_price_df.set_index('index')
ng_price

,CH4
index,
2020-01-01 00:00:00,0.093304
2020-01-02 00:00:00,0.091518
2020-01-03 00:00:00,0.091964
2020-01-04 00:00:00,0.091964
2020-01-05 00:00:00,0.091964
...,...
2020-12-27 00:00:00,0.119643
2020-12-28 00:00:00,0.119643
2020-12-29 00:00:00,0.106696


**import cost data**


In [5]:
cost_dict = get_data(file_name='data/cost_dict')
for i in cost_dict['HO']['moderate'].keys():
    print(i + ':', cost_dict['HO']['moderate'][i]['0'])

LiI_c: {'CAPEX': 1302181.81818181, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 0, 'units': '$/MW', 'source': 'NREL Annual Technology Baseline 2021, https://atb.nrel.gov/'}
LiI_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
CAES_c: {'CAPEX': 1669000.0, 'Fixed O&M': 16700.0, 'Variable O&M': 0, 'units': '$/MW', 'source': 'https://www.pnm.com/documents/396023/1506047/2017+-+HDR+10-30-17+PNM+Energy+Storage+Report.pdf/a2b7ca65-e1ba-92c8-308a-9a8391a87331'}
CAES_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
PSH_c: {'CAPEX': 0, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 4435.188, 'units': '$/MW', 'source': 'NREL Annual Technology Baseline 2021, https://atb.nrel.gov/'}
PSH_d: {'CAPEX': 0.001, 'Fixed O&M': 0.001, 'Variable O&M': 0, 'units': '$/MW', 'source': 'Dummy Process'}
PV: {'CAPEX': 1302181.81818181, 'Fixed O&M': 41432.7272727272, 'Variable O&M': 0, 'units': '$/MW', 'sour

$\textbf{Define temporal scale}$


In [6]:
scales = Temporal_scale(discretization_list=[1, 365, 24], start_zero= 2020)

$\textbf{Declare constants for ease}$


In [7]:
bigM = 10**4  # very large number
smallM = 0.1
water_price = 31.70  # $/5000gallons
power_price = 8  # cents/kWh
ur_price = 42.70  # 250 Pfund U308 (Uranium)
A_f = 0.05  # annualization factor
# CO2_res = 0.2
pv_start = 0
ake_start = 0
smrh_start = 0
smr_start = 0
asmr_start = 0

$\textbf{Declare resources}$

In [8]:
Charge = Resource(name='Charge', sell=False,
                  store_max=100, basis='MW', label='Battery energy', block='energystorage')

Air_C = Resource(name='Air_C', store_max=bigM, basis='MW',
                 label='CAES energy', block='energystorage')

H2O_E = Resource(name='H2O_E', store_max=bigM, basis='MW',
                 label='PSH energy', block='energystorage')

Uranium = Resource(name='Uranium', cons_max=(1/4.17*10**(-5))*bigM,
                   price=ur_price/(250/2), basis='kg', label='Uranium', block='energyfeedstock')

Solar = Resource(
    name='Solar', cons_max=bigM, basis='MW', label='Solar Power', block='energyfeedstock')

Wind = Resource(name='Wind', cons_max= bigM, basis='MW', label='Wind Power', block='energyfeedstock')

H2_L = Resource(name='H2_L', store_max=10**10, revenue=2,
                mile=1/(0.1180535*1.60934), basis='kg', label='Hydrogen - Geological', block='resourcestorage')

H2_C = Resource(name='H2_C', store_max= 10**10, loss=0.025/24, revenue=2, mile=1/(0.1180535*1.60934), \
    basis='kg', label='Hydrogen - Local Cryo', block='resourcestorage')


# H2 = Resource(name='H2', basis='kg', sell = True, demand = True, label='Hydrogen', block='Resource')
H2 = Resource(name='H2', basis='kg', label='Hydrogen', block='Resource')


H2O = Resource(name='H2O', cons_max=10**10,
               price=water_price/(5000*3.7854), basis='kg', label='Water', block='Resource')
            
O2 = Resource(name='O2', sell=True, loss=0.07,
              basis='kg', label='Oxygen', block='Resource')


CH4 = Resource(name='CH4', cons_max=10 **
               10, price=0.113891, basis='kg', label='Natural gas', block='materialfeedstock')

CO2 = Resource(name='CO2', basis='kg',
               label='Carbon dioxide', block='Resource')


CO2_DAC = Resource(
    name='CO2_DAC', basis='kg', label='Carbon dioxide - captured', block='carbonsequestration')

CO2_AQoff = Resource(
    name='CO2_AQoff', store_max=10**6, basis='kg', label='Carbon dioxide - sequestered', block='carbonsequestration')

CO2_EOR = Resource(
    name='CO2_EOR', store_max=10**6, basis='kg', label='Carbon dioxide - EOR', block='carbonsequestration')


CH3OH = Resource(name='CH3OH', sell=True, revenue=0.5,
                 mile=1/(0.0195508*1.60934), basis='kg', label='Methanol', block='resourcedischarge')

CO2_Vent = Resource(
    name='CO2_Vent', sell=True, basis='kg', label='Carbon dioxide - Vented', block='resourcedischarge')

# Power= Resource(name= 'Power', sell= True, store_max=0,   \
#    mile= (10**3)/(0.2167432**1.60934), label= 'Renewable power generated')

Power = Resource(name='Power', basis='MW',
                 label='Renewable power generated', block='Resource')

Mile = Resource(name = 'Mile', basis = 'miles', sell = True, demand  = True, label = 'miles driven')
Geothermal_energy= Resource(name='Geothermal', store_max=10**10, sell=True, basis='MW',  label='Geothermal Storage', block='resourcestorage')
Biogas = Resource(name = 'Biogass', basis = 'kg', cons_max = 10**5, label = 'Biogas consumed from outside the system' )
Biomass = Resource(name = 'Biomass', basis = 'kg', cons_max = 10**5, label = 'Biomass consumed from outside the system' )


$\textbf{Declare Materials}$


In [9]:
LiR = Material(name='LiR', gwp=1.484, resource_cons = {H2O: 2273}, toxicity=793, basis= 'kg', label='Rock-based Lithium', citation= 'Nelson Bunyui Manjong (2021), httoxicitys://www.tcc.fl.edu/media/divisions/academic-affairs/academic-enrichment/urc/poster-abstracts/Xanders_Madison_Poster_URS.pdf') #gwp=(0.216,0.314)
#several animals
LiB = Material(name='LiB', gwp=0.031, toxicity=793, basis= 'kg', label='Brine-based Lithium', citation= 'Nelson Bunyui Manjong (2021)') #gwp=(0.289,0.499)
Mn = Material(name='Mn', gwp=4.51, toxicity=1484, basis= 'kg', label='Manganese', citation= 'Nelson Bunyui Manjong (2021)') #inventory data #9.6, 2.6,7.9
Ni = Material(name='Ni', gwp=7.64, resource_cons = {H2O: 80}, toxicity=67, basis= 'kg', label='Nickel', citation= 'Mark Mistry (2016), Sustainable water use in minerals and metal production')
Co = Material(name='Co', gwp=11.73, toxicity=325, basis= 'kg', label='Cobalt', citation= 'Farjana et al. 2019a, b')
# Steel = Material(name='Steel', gwp=(0.8,1.4), resource_cons = {H2O: 3.94}, toxicity=40, basis= 'kg', label='Steel', citation= 'Kim R.Bawden (2016), Anlong Li (2020)') #toxicity of iron instead
Steel = Material(name='Steel', gwp=0.8, resource_cons = {H2O: 3.94}, toxicity=40, basis= 'kg', label='Steel', citation= 'Kim R.Bawden (2016), Anlong Li (2020)') #toxicity of iron instead

# Fe = Material(name='Steel', gwp=(0.8,1.4), resource_cons = {H2O: 3.94}, toxicity=40, basis= 'kg', label='Iron', citation= 'Kim R.Bawden (2016), Anlong Li (2020)')
Fe = Material(name='Steel', gwp=0.8, resource_cons = {H2O: 3.94}, toxicity=40, basis= 'kg', label='Iron', citation= 'Kim R.Bawden (2016), Anlong Li (2020)')
CuP = Material(name='Cu', gwp=2.5, resource_cons = {H2O: 30}, toxicity=2500, basis= 'kg', label='Copper Pyro', citation= 'T.E. Norgate (2007), Sustainable water use in minerals and metal production')
CuH = Material(name='Cu', gwp=6, resource_cons = {H2O: 40}, toxicity=2500, basis= 'kg', label='Copper Hyro', citation= 'T.E. Norgate (2007), Sustainable water use in minerals and metal production')
PbB = Material(name='PbB', gwp=1.5, resource_cons = {H2O: 15}, toxicity=450, basis= 'kg', label='Lead BF', citation= 'T.E. Norgate (2007), Sustainable water use in minerals and metal production')
PbI = Material(name='PbI', gwp=2.5, resource_cons = {H2O: 25}, toxicity=450, basis= 'kg', label='Lead ISF', citation= 'T.E. Norgate (2007), Sustainable water use in minerals and metal production')
ZnE = Material(name='ZnE', gwp=5, resource_cons = {H2O: 30}, toxicity=57, basis= 'kg', label='Zn Elec', citation= 'T.E. Norgate (2007), Sustainable water use in minerals and metal production')
ZnI = Material(name='ZnI', gwp=3.5, resource_cons = {H2O: 25}, toxicity=57, basis= 'kg', label='Zn ISF', citation= 'T.E. Norgate (2007), Sustainable water use in minerals and metal production')
Pd = Material(name='Pd', gwp=3880, resource_cons = {H2O: 210713}, toxicity=3, basis= 'kg', label='Palladium', citation= 'Philip Nuss (2014), Simon Meißner (2021)')
Rd = Material(name='Rd', gwp=35100, toxicity=15, basis= 'kg', label='Rhodium', citation= 'Philip Nuss (2014)')
Pt = Material(name='Pd', gwp=12500, resource_cons = {H2O: 31349}, toxicity=5000, basis= 'kg', label='Platinum', citation= 'Philip Nuss (2014), Simon Meißner (2021)')
Y = Material(name='Y', gwp=15.1, toxicity=400, basis= 'kg', label='Yttrium', citation= 'Philip Nuss (2014)')
Al = Material(name='Al', gwp=8.2, resource_cons = {H2O: 147}, toxicity=3632, basis= 'kg', label='Aluminium', citation= 'Philip Nuss (2014), Water requirements of the aluminum industry Water Supply Paper 1330-C')
Mg = Material(name='Mg', gwp=9.6, toxicity=200, basis= 'kg', label='Magnesium', citation= 'Philip Nuss (2014)')
PP = Material(name='PP', gwp=1.586, basis= 'kg', label='Polypropylene', citation= 'Greenhouse gas emissions and natural capital implications of plastics (including biobased plastics)')
HDPE = Material(name='HDPE', gwp=1.98,toxicity=5000, basis= 'kg', label='HD polyethylene', citation= 'Greenhouse gas emissions and natural capital implications of plastics (including biobased plastics)')
LDPE = Material(name='LDPE', gwp=1.93, toxicity=2000, basis= 'kg', label='LD polyethylene', citation= 'Greenhouse gas emissions and natural capital implications of plastics (including biobased plastics)')
PVC = Material(name='PVC', gwp=2.51, basis= 'kg', label='Polyvinylchloride', citation= 'Greenhouse gas emissions and natural capital implications of plastics (including biobased plastics)')
Rubber = Material(name='Rubber', gwp=6.4, toxicity=330, basis= 'kg', label='Rubber', citation= 'GREENING OF INDUSTRY NETWORK (GIN) 2010: CLIMATE CHANGE AND GREEN GROWTH: INNOVATING FOR SUSTAINABILITY ')
PTFE = Material(name='PTFE', gwp=9.6, toxicity=11280, basis= 'kg', label='Poly tetra fluoroethylene', citation='httoxicitys://shamrocktechnologies.com/co2-emissions/')
# Epoxy = Material(name='Epoxy', gwp=(4.7, 8.1), toxicity=2000 , basis= 'kg', label='Epoxy resin', citation= 'Bricout et al. (2017)' )
Epoxy = Material(name='Epoxy', gwp=4.7, toxicity=2000 , basis= 'kg', label='Epoxy resin', citation= 'Bricout et al. (2017)' )

Polyester = Material(name='Polyester', gwp=4.32, toxicity=2500, basis= 'kg', label='Polyester resin', citation= ' Rietveld and Hegger (2014)' )
Vinyl_easter = Material(name='vinyl ester', gwp=5.87, toxicity=1000, basis= 'kg', label='vinyl ester resin', citation= ' Rietveld and Hegger (2014) ' ) #Assume toxicity
Glass_fibre = Material(name='glass fibre', gwp=2.5, toxicity=1000, basis= 'kg', label='Glass fibre resin', citation= ' Bachmann et al. (2017)' ) #Assume toxicity



$\textbf{Declare processes}$

In [10]:
LiI_c = Process(name='LiI_c', conversion={Charge: 1, Power: -1}, cost = cost_dict['HO']['moderate']['LiI_c']['0'],\
    prod_max=100, trl='nrel', block='power_storage', label='Lithium-ion battery', citation='Zakeri 2015')

LiI_d = Process(name='LiI_d', conversion={Charge: -1.1765, Power: 1}, cost =  {'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': smallM, \
    'units': '$/kg','source': 'dummy'}, \
    prod_max=100, trl='discharge', block='power_storage', label='Lithium-ion battery discharge', citation='Zakeri 2015')

CAES_c = Process(name='CAES_c', conversion={Air_C: 1, Power: -1}, cost = cost_dict['HO']['moderate']['CAES_c']['0'], \
    intro_scale=0, prod_max=bigM, trl='pilot', block='power_storage', label='Compressed air energy storage (CAES)', citation='Zakeri 2015')

# CAES_d = Process(name='CAES_d', conversion={Air_C: -1.4286, Power: 1}, cost =  {'CAPEX': 1669, 'Fixed O&M': 22, 'Variable O&M': [2, 120], \
#     'units': '$/kg','source': 'dummy'},\
#     intro_scale=0, prod_max=bigM, trl='discharge', block='power_storage', label='Compressed air energy storage (CAES) discharge', citation='Zakeri 2015')

CAES_d = Process(name='CAES_d', conversion={Air_C: -1.4286, Power: 1}, cost =  {'CAPEX': 1669, 'Fixed O&M': 22, 'Variable O&M': 2, \
    'units': '$/kg','source': 'dummy'},\
    intro_scale=0, prod_max=bigM, trl='discharge', block='power_storage', label='Compressed air energy storage (CAES) discharge', citation='Zakeri 2015')


PSH_c = Process(name='PSH_c', conversion={H2O_E: 1, Power: -1}, cost = cost_dict['HO']['moderate']['PSH_c']['0'], \
    intro_scale=0, prod_max=bigM, trl='nrel', block='power_storage', label='Pumped storage hydropower (PSH)', citation='Zakeri 2015')

# PSH_d = Process(name='PSH_d', conversion={H2O_E: -1.4286, Power: 1}, cost =  {'CAPEX': 2638, 'Fixed O&M': 3, 'Variable O&M': [5, 100], \
#     'units': '$/kg','source': 'dummy'}, material_cons = {Glass_fibre: 100},\
#     prod_max=bigM, trl='discharge', block='power_storage', label='Pumped storage hydropower (PSH) discharge', citation='Zakeri 2015')


PSH_d = Process(name='PSH_d', conversion={H2O_E: -1.4286, Power: 1}, cost =  {'CAPEX': 2638, 'Fixed O&M': 3, 'Variable O&M': 5, \
    'units': '$/kg','source': 'dummy'}, material_cons = {Glass_fibre: 100},\
    prod_max=bigM, trl='discharge', block='power_storage', label='Pumped storage hydropower (PSH) discharge', citation='Zakeri 2015')


WF = Process(name='WF', conversion={Wind: -1, Power: 1, H2O: -1}, cost=cost_dict['HO']['moderate']['WF']['0'],
             prod_max=100, gwp=52700, land=10800/1800, trl='nrel', block='power_generation', material_cons = {Steel: 73, Fe:11, CuP: 1, Al: 1, Epoxy: 4, Polyester: 4, Vinyl_easter: 3, Glass_fibre: 3},
             label='Wind mill array', citation='Use windtoolkit conversion') #material is percentage


PV = Process(name='PV', intro_scale=pv_start, conversion={Solar: -1, Power: 1, H2O: -20}, cost=cost_dict['HO']['moderate']['PV']['0'],
             prod_max=100, gwp=53000, land=13320/1800, trl='nrel', block='power_generation', \
                 label='Solar photovoltaics (PV) array', citation='Use pvlib conversion')

AKE = Process(name='AKE', intro_scale=ake_start, conversion={Power: -1, H2: 19.474, O2: 763.2, H2O: -175.266},
              cost=cost_dict['HO']['moderate']['AKE']['0'], prod_max=bigM, trl='utility', block='material_production',
              label='Alkaline water electrolysis (AWE)', citation='Demirhan et al. 2018 AIChE paper')  # 20.833 MW required to produce 1000t/day.H2

SMRH = Process(name='SMRH', intro_scale=smrh_start, conversion={Power: -1.11*10**(-3), CH4: -3.76, H2O: -23.7, H2: 1, CO2_Vent: 1.03, CO2: 9.332},
               cost=cost_dict['HO']['moderate']['SMRH']['0'], prod_max=bigM, gwp=0, trl='enterprise', block='material_production',
               label='Steam methane reforming + CCUS', citation='Mosca 2020, 90pc capture')

SMR = Process(name='SMR', intro_scale=smr_start, cost= {'CAPEX': 2400, 'Fixed O&M': 800, 'Variable O&M': 0.03, 'units': '$/kg', 'source': 'dummy'}, \
    conversion={Power: -1.11*10**(-3), CH4: -3.76, H2O: -23.7, H2: 1, CO2_Vent: 9.4979}, prod_max=bigM, gwp=0, trl='enterprise',
                      block='material_production', label='Steam methane reforming', citation='Mosca 2020')

ASMR = Process(name='ASMR', conversion={Uranium: -4.17*10**(-5), H2O: -3364.1, Power: 1}, cost=cost_dict['HO']['moderate']['ASMR']['0'],
               intro_scale=asmr_start, gwp=9100, prod_max=bigM, land=1100/1800, trl='pilot', block='power_generation', label='Small modular reactors (SMRs)')

H2_C_c = Process(name='H2_C_c', conversion={Power: -1.10*10**(-3), H2_C: 1, H2: -1}, cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 0,
                                                                                           'units': '$/kg', 'source': 'dummy'},
                 prod_max=12000, gwp=0, trl='pilot', block='material_storage', label='Hydrogen local storage (Compressed)',
                 citation='Bossel and Eliasson - Energy and the Hydrogen Economy')

H2_C_d = Process(name='H2_C_d',  conversion={H2_C: -1, H2: 1}, cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 0,
                                                                     'units': '$/kg', 'source': 'dummy'},
                 prod_max=bigM, gwp=0, trl='nocost',
                 block='material_storage', label='Hydrogen local storage (Compressed) discharge', citation='Bossel and Eliasson - Energy and the Hydrogen Economy')

H2_L_c = Process(name='H2_L_c', conversion={Power: -4.17*10**(-4), H2_L: 1, H2: -1}, cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 0,
                                                                                           'units': '$/kg', 'source': 'dummy'},
                 prod_max=bigM, gwp=0, trl='repurposed', block='material_storage', label='Hydrogen geological storage',
                 citation='Bossel and Eliasson - Energy and the Hydrogen Economy')

H2_L_d = Process(name='H2_L_d', conversion={H2_L: -1, H2: 1}, prod_max=bigM, gwp=0, trl='nocost', cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 0,
                                                                                                        'units': '$/kg', 'source': 'dummy'},
                 block='material_storage', label='Hydrogen geological storage discharge', citation='Bossel and Eliasson - Energy and the Hydrogen Economy')

DAC = Process(name='DAC', conversion={Power: -1.93*10**(-4), H2O: -4.048, CO2_DAC: 1}, cost = cost_dict['HO']['moderate']['DAC']['0'], \
    intro_scale=4, prod_max=bigM, gwp=0, trl='pilot', block='CCUS', lifetime=(12,20), label='Direct air capture', citation='D. Belloti et al (2017), Beuttler (2019)')

EOR = Process(name='EOR', intro_scale=0, conversion={Power: -0.00255, CO2: -1, CO2_EOR: 1, CO2_Vent: 0.67}, \
    cost = cost_dict['HO']['moderate']['EOR']['0'], prod_max=bigM, gwp=438, carbon_credit=True, \
        trl='enterprise', block='CCUS', label='CO2-Enhanced oil recovery', citation='Nicholas A.Azzolina (2016)') #unit for gwp is per bbl

AQoff_SMR = Process(name='AQoff_SMR', conversion={Power: -0.00128, CO2_AQoff: 1, CO2: -1}, cost=cost_dict['HO']['moderate']['AQoff_SMR']['0'],
                    prod_max=bigM, carbon_credit=True, trl='repurposed', block='CCUS', label='Offshore aquifer CO2 sequestration (SMR)')

# EV = Process(name = 'EV', conversion = {Power: -(0.2167432*10**(-3))/1.60934, Mile: 1}, prod_max= bigM, gwp = 0, \
#     material_cons = {LiB: 2.1, LiR: 5.9, Ni: 35, Co: 14, Mn: 20, PP: 48, PVC: 22.50, HDPE: 25.5}, \
#         cost={'CAPEX': 271, 'Fixed O&M': 4.455, 'Variable O&M': [600, 3800], 'units': '$/kW, $/kW-yr, $/KWH', 'source': '38, 12, 39, 36'}, \
#         citation = 'https://www.nature.com/articles/d41586-021-02222-1', label = 'electric vehicle') #material is per EV, set a random combination for total Li

EV = Process(name = 'EV', conversion = {Power: -(0.2167432*10**(-3))/1.60934, Mile: 1}, prod_max= bigM, gwp = 0, \
    material_cons = {LiB: 2.1, LiR: 5.9, Ni: 35, Co: 14, Mn: 20, PP: 48, PVC: 22.50, HDPE: 25.5}, \
        cost={'CAPEX': 271, 'Fixed O&M': 4.455, 'Variable O&M': 600, 'units': '$/kW, $/kW-yr, $/KWH', 'source': '38, 12, 39, 36'}, \
        citation = 'https://www.nature.com/articles/d41586-021-02222-1', label = 'electric vehicle') #material is per EV, set a random combination for total Li


HV = Process(name = 'HV', conversion = {H2: -0.0195504/1.60934, Mile: 1}, prod_max= bigM, gwp = 0, \
    material_cons = {LiB: 2.1, LiR: 5.9, Ni: 35, Co: 14, Mn: 20, PP: 48, PVC: 22.50, HDPE: 25.5},
    cost={'CAPEX': 2823, 'Fixed O&M': 28.51, 'Variable O&M': 15, 'units': '$/kg', 'source': '39, 36'}, label = 'hydrogen vehicle')

MV = Process(name = 'HV', conversion = {CH3OH: -0.1180535/1.60934, Mile: 1}, prod_max= bigM, gwp = 0, \
    cost={'CAPEX': smallM, 'Fixed O&M': 0, 'Variable O&M': 1, 'units': '$/kg', 'source': 'dummy'}, label = 'methanol vehicle')

Na_S_c = Process(name='Na-S', conversion={Charge: 1, Power: -1},  trl='utility', gwp=0.028, lifetime=(10,15), label='Sodium-Sulfur', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Lead_a_c = Process(name='Lead',  conversion={Charge: 1, Power: -1}, trl='utility', material_cons = {PbB: 20}, gwp=0.075, lifetime=(5,15), label='Lead-acid', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Lead_b_c = Process(name='Lead',  conversion={Charge: 1, Power: -1}, trl='utility', material_cons = {PbI: 20}, gwp=0.075, lifetime=(5,15), label='Lead-acid', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Na_S_d = Process(name='Na-S', conversion={Charge: -1, Power: 1}, label='Sodium-Sulfur', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Lead_a_d = Process(name='Lead',  conversion={Charge: -1, Power: 1}, label='Lead-acid', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Lead_b_d = Process(name='Lead',  conversion={Charge: -1, Power: 1},label='Lead-acid', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')

Vrb_c = Process(name='VRB', conversion={Charge: 1, Power: -1},trl='utility', gwp=0.02, lifetime=(5,10), label='VRB', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Ni_Cd_c = Process(name='Na-S', conversion={Charge: 1, Power: -1},trl='utility', material_cons = {Ni: 20}, gwp=0.08, lifetime=(10,20), label='Ni-Cd', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')

Vrb_d = Process(name='VRB',  conversion={Charge: -1, Power: 1}, label='VRB', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')
Ni_Cd_d = Process(name='Na-S',  conversion={Charge: -1, Power: 1},label='Ni-Cd', citation= 'Florin, N. and Dominish, E. (2017), Zakeri 2015')

CO2_based_methanol = Process(name='methanol', conversion = {CO2: -1, CH3OH: 1}, trl='pilot', gwp=-1.7, label='CO2 CH3OH', citation= 'muller (2020)')
Biogas_plant = Process(name='Biogas', trl='pilot', conversion = {Biogas:-1, Power: 1}, gwp=-0.42, lifetime=20, label='Biogas', citation= 'Economic and Global Warming Potential Assessment of Flexible Power Generation with Biogas Plants, Life Cycle Environmental Impacts of Electricity from Biogas Produced by Anaerobic Digestion')
Biomass = Process(name='Biomass', trl='pilot', conversion = {Biomass: -1, Power: 1}, gwp=0.13, label='Biomass', citation= 'Analysis of the Global Warming Potential of Biogenic CO2 Emission in Life Cycle Assessments') #gwp=0.049 nrel

# CO2_based_methanol = Process(name='methanol', conversion = {CO2: -1, CH3OH: 1}, trl='pilot', gwp=(-1.7,9.7), label='CO2 CH3OH', citation= 'muller (2020)')
# Biogas_plant = Process(name='Biogas', trl='pilot', conversion = {Biogas:-1, Power: 1}, gwp=(-0.42, 0.06), lifetime=20, label='Biogas', citation= 'Economic and Global Warming Potential Assessment of Flexible Power Generation with Biogas Plants, Life Cycle Environmental Impacts of Electricity from Biogas Produced by Anaerobic Digestion')
# Biomass = Process(name='Biomass', trl='pilot', conversion = {Biomass: -1, Power: 1}, gwp=(0.13,0.32), label='Biomass', citation= 'Analysis of the Global Warming Potential of Biogenic CO2 Emission in Life Cycle Assessments') #gwp=0.049 nrel

In [11]:
# process_set = {LiI_c, LiI_d, CAES_c, CAES_d, PSH_c, PSH_d, WF, PV, AKE, SMRH, SMR, ASMR, H2_C_d, \
#     H2_C_d, H2_L_c, H2_L_d, DAC, EOR, AQoff_SMR, EV, HV, MV, Na_S_c, Lead_a_c, Lead_b_c, Vrb_c,  \
#         Ni_Cd_c, Na_S_d, Lead_a_d, Lead_b_d, Vrb_d, Ni_Cd_d, CO2_based_methanol, Biogas_plant, Biomass}

process_set = {LiI_c, LiI_d, DAC, WF, PV, AKE, H2_L_c, H2_L_d, EV, Na_S_c, Lead_a_c, Lead_b_c, Vrb_c,  \
        Ni_Cd_c, Na_S_d, Lead_a_d, Lead_b_d, Vrb_d, Ni_Cd_d, CO2_based_methanol, Biogas_plant, Biomass} #  HV, MV,

$\textbf{Declare location(s)}$


In [12]:
HO = Location(name='HO', processes= process_set, demand_factor= {Mile: ercot}, \
    cost_factor = {CH4: ng_price}, capacity_factor = {PV: pandas.DataFrame(weather['dni']), WF: pandas.DataFrame(weather['wind_speed'])}, scales=scales, \
        label='Houston', demand_level=2, capacity_level= 2, cost_level= 1)

In [13]:
# plot.capacity_factor(location= HO, process= PV, color= 'orange')
# plot.capacity_factor(location= HO, process= WF, color= 'blue')
# plot.cost_factor (location= HO, resource= CH4, color= 'red')
# plot.demand_factor (location= HO, resource= Mile)

In [14]:
scenario = Scenario(name= 'shell', network= HO, scales= scales,  expenditure_scale_level= 2, scheduling_scale_level= 2, \
    network_scale_level= 0, demand_scale_level= 2, label= 'shell milp case study (HO)')

$\textbf{Plot varying data input}$

In [15]:
# plot.capacity_factor(location= HO_reduced, process= PV, color= 'orange')
# plot.capacity_factor(location= HO_reduced, process= WF, color= 'blue')
# plot.cost_factor (location= HO_reduced, resource= CH4, color= 'red')
# plot.demand_factor (location= HO_reduced, resource= Mile)

$\textbf{Formulate model}$

A pyomo instance is formulated from the scenario

Concises sets and corresponding variables are declared.

Corresponding constraints are generated based on the nature of model chosen

In the presented example, a MILP is formulated



In [16]:
milp = formulate(scenario= scenario, demand = 1000, \
    constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.land, Constraints.resource_balance, Constraints.emission}, \
        objective= Objective.cost)

In [17]:
results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
    name=f"Material_case_study")

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (mac64[x86])
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads
Optimize a model with 368121 rows, 271777 columns and 1069125 nonzeros
Model fingerprint: 0xb01e725b
Variable types: 271745 continuous, 32 integer (32 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+10]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e+02, 1e+10]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve removed 289148 rows and 210324 columns
Presolve time: 1.69s
Presolved: 78973 rows, 61453 columns, 226947 nonzeros
Variable types: 61446 continuous, 7 integer (7 binary)

Deterministic concurrent LP optimizer: primal and dual simplex
Showing first log only...


Root simplex log...

Iteration    Objective       Primal Inf.    Dual Inf.      Time
   12316    3.1486453e+09   0.000000e+00   3.568158e+07      5s
   20336    3.1486

In [18]:
results.output

{'termination': 'optimal',
 'LB': 3148610428.0751996,
 'UB': 3148610428.0751996,
 'n_cons': 368121,
 'n_vars': 271777,
 'n_binvars': 32,
 'n_intvars': 32,
 'n_convars': 271713,
 'n_nonzero': 1069125,
 'P': {('HO', 'Biomass', 0, 0, 0): 0.0,
  ('HO', 'Biomass', 0, 0, 1): 0.0,
  ('HO', 'Biomass', 0, 0, 2): 0.0,
  ('HO', 'Biomass', 0, 0, 3): 0.0,
  ('HO', 'Biomass', 0, 0, 4): 0.0,
  ('HO', 'Biomass', 0, 0, 5): 0.0,
  ('HO', 'Biomass', 0, 0, 6): 0.0,
  ('HO', 'Biomass', 0, 0, 7): 0.0,
  ('HO', 'Biomass', 0, 0, 8): 0.0,
  ('HO', 'Biomass', 0, 0, 9): 0.0,
  ('HO', 'Biomass', 0, 0, 10): 0.0,
  ('HO', 'Biomass', 0, 0, 11): 0.0,
  ('HO', 'Biomass', 0, 0, 12): 0.0,
  ('HO', 'Biomass', 0, 0, 13): 0.0,
  ('HO', 'Biomass', 0, 0, 14): 0.0,
  ('HO', 'Biomass', 0, 0, 15): 0.0,
  ('HO', 'Biomass', 0, 0, 16): 0.0,
  ('HO', 'Biomass', 0, 0, 17): 0.0,
  ('HO', 'Biomass', 0, 0, 18): 0.0,
  ('HO', 'Biomass', 0, 0, 19): 0.0,
  ('HO', 'Biomass', 0, 0, 20): 0.0,
  ('HO', 'Biomass', 0, 0, 21): 0.0,
  ('HO', 'Bio

In [19]:
def gwp_reduce_solve(gwp_reduction_pct, gwp):
    milp = formulate(scenario= scenario, gwp_reduction_pct= gwp_reduction_pct, gwp= gwp)
    results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
        name=f"Material_case_study")
    return results

In [20]:
results_dict = {i: gwp_reduce_solve(gwp_reduction_pct=2*i, gwp = 718419.21) for i in range(4)}

TypeError: formulate() missing 2 required positional arguments: 'constraints' and 'objective'

In [ ]:
results.output['Capex_location']

In [ ]:
results.output['Capex_location']

In [ ]:
results.output['Vopex_location']

In [ ]:
results.output['Fopex_location']

In [ ]:
results.output['Cap_P']

In [ ]:
results.output['X_P']


In [ ]:
results.output['global_warming_potential_resource']

In [ ]:
results.output['global_warming_potential_network']
